# ViT-B/16 — Vision Transformer (ImageNet-21k)

**EEEM068 Applied Machine Learning — University of Surrey**

Architecture experiment using **ViT-B/16 pre-trained on ImageNet-21k** (supervised, via `timm`).  
Compared against the EfficientNet-B0 baseline under identical training conditions.

| Setting | Value |
|---|---|
| Architecture | ViT-B/16 (`vit_base_patch16_224.augreg_in21k_ft_in1k`) |
| Pre-training | ImageNet-21k supervised → fine-tuned on ImageNet-1k |
| Loss | CrossEntropyLoss (no class weights) |
| Sampler | Standard shuffle |
| Augmentation | Resize → RandomCrop → HFlip → Normalise |
| Normalisation | mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5) |
| Optimiser | AdamW, lr=3e-4, weight_decay=1e-2 |
| Scheduler | CosineAnnealingLR |
| Metric | Quadratic Weighted Kappa (QWK) |

## 1. Imports

In [19]:
import os
import random
from pathlib import Path

import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import (
    classification_report, cohen_kappa_score, confusion_matrix
)
from sklearn.metrics import f1_score, accuracy_score

print(f'PyTorch : {torch.__version__}')
print(f'timm    : {timm.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')

PyTorch : 2.11.0+cu130
timm    : 1.0.26
CUDA    : True


## 2. Configuration

In [ ]:
# ── Paths ─────────────────────────────────────────────────────
NOTEBOOK_DIR    = Path().resolve()
ROOT            = NOTEBOOK_DIR.parent

DATA_ROOT       = ROOT / 'data' / 'diabetic-retinopathy-detection'
TRAIN_DIR       = DATA_ROOT / 'train_proc'        # 13-stage preprocessed images
VAL_DIR         = DATA_ROOT / 'val_proc'
TEST_LABELS_DIR = DATA_ROOT / 'test_labels_proc'
TRAIN_CSV       = DATA_ROOT / 'train_split.csv'
VAL_CSV         = DATA_ROOT / 'val_split.csv'
TEST_CSV        = DATA_ROOT / 'test_split.csv'

OUT_DIR     = ROOT / 'outputs' / 'vit_b16'
CKPT_DIR    = OUT_DIR / 'checkpoints'
FIGURES_DIR = OUT_DIR / 'figures'
for d in (OUT_DIR, CKPT_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ── Experiment ────────────────────────────────────────────────
EXPERIMENT_NAME = 'vit_b16'
MODEL_NAME      = 'vit_base_patch16_224.augreg_in21k_ft_in1k'

# ── Reproducibility ───────────────────────────────────────────
SEED = 42

# ── Data ──────────────────────────────────────────────────────
IMAGE_SIZE  = 224
NUM_CLASSES = 5
CLASS_NAMES = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']

# ViT-B/16 (IN21k) uses symmetric normalisation, not ImageNet RGB stats
VIT_MEAN = (0.5, 0.5, 0.5)
VIT_STD  = (0.5, 0.5, 0.5)

# ── Training ──────────────────────────────────────────────────
BATCH_SIZE               = 32
NUM_WORKERS              = 4
EPOCHS                   = 10
LR                       = 3e-4
WEIGHT_DECAY             = 1e-2
EARLY_STOPPING_PATIENCE  = 3

print('Config set. Root:', ROOT)
print(f'Images: {TRAIN_DIR.name} / {VAL_DIR.name} / {TEST_LABELS_DIR.name}')

In [21]:
# ── Seed + device ─────────────────────────────────────────────
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

Device: cuda


In [ ]:
# ── Checkpoint paths — defined once here, reused by every eval cell ──────────
# Running this cell is sufficient to resume evaluation after a kernel restart.
CKPT_BASELINE = CKPT_DIR / f'{EXPERIMENT_NAME}_best.pt'
CKPT_EXP1     = ROOT / 'outputs' / 'vit_b16_exp1' / 'checkpoints' / 'vit_b16_exp1_best.pt'
CKPT_EXP2     = ROOT / 'outputs' / 'vit_b16_exp2' / 'checkpoints' / 'vit_b16_exp2_best.pt'
CKPT_EXP3_P1  = ROOT / 'outputs' / 'vit_b16_exp3' / 'checkpoints' / 'vit_b16_exp3_phase1_best.pt'
CKPT_EXP3     = ROOT / 'outputs' / 'vit_b16_exp3' / 'checkpoints' / 'vit_b16_exp3_best.pt'
CKPT_EXP5     = ROOT / 'outputs' / 'vit_b16_exp5' / 'checkpoints' / 'vit_b16_exp5_best.pt'

print('Checkpoint registry:')
for label, path in [
    ('Baseline',  CKPT_BASELINE),
    ('Exp1',      CKPT_EXP1),
    ('Exp2',      CKPT_EXP2),
    ('Exp3 P1',   CKPT_EXP3_P1),
    ('Exp3',      CKPT_EXP3),
    ('Exp5',      CKPT_EXP5),
]:
    status = 'saved' if path.exists() else 'not yet trained'
    print(f'  {label:<12}: {path.relative_to(ROOT)}  [{status}]')


## 3. Load Pre-Split Data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print(f'Train : {len(train_df):,} images  (dir: train/)')
print(f'Val   : {len(val_df):,} images   (dir: val/)')
print(f'Test  : {len(test_df):,} images   (dir: test_labels/)')
print()
print(f'{"Grade":<6} {"Name":<18} {"Train":>7} {"Val":>7} {"Test":>7}')
print('-' * 52)
for g, name in enumerate(CLASS_NAMES):
    t = (train_df['level'] == g).sum()
    v = (val_df['level']   == g).sum()
    s = (test_df['level']  == g).sum()
    print(f'  {g}    {name:<18} {t:>6,}  {v:>6,}  {s:>6,}')

## 4. Dataset and DataLoaders

In [ ]:
class DRDataset(Dataset):
    """Fundus image dataset. Reads images from data_dir using the 'image' column."""

    def __init__(self, df: pd.DataFrame, data_dir: Path, transform=None):
        self.df        = df.reset_index(drop=True)
        self.data_dir  = Path(data_dir)
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int):
        row   = self.df.iloc[idx]
        image = Image.open(self.data_dir / f"{row['image']}.jpeg").convert('RGB')
        label = int(row['level'])
        if self.transform is not None:
            image = self.transform(image)
        return image, label


# ViT requires symmetric normalisation (0.5, 0.5, 0.5) — different from EfficientNet
train_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.14)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(VIT_MEAN, VIT_STD),
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(VIT_MEAN, VIT_STD),
])

train_ds = DRDataset(train_df, TRAIN_DIR,       transform=train_transform)
val_ds   = DRDataset(val_df,   VAL_DIR,         transform=val_transform)
test_ds  = DRDataset(test_df,  TEST_LABELS_DIR, transform=val_transform)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
    persistent_workers=NUM_WORKERS > 0,
)

print(f'Train loader: {len(train_loader)} batches/epoch')
print(f'Val loader  : {len(val_loader)} batches/epoch')
print(f'Test loader : {len(test_loader)} batches/epoch')

## 5. Model

`vit_base_patch16_224.augreg_in21k_ft_in1k` — pre-trained on ImageNet-21k (21,843 classes),  
then fine-tuned on ImageNet-1k. The 1000-class head is replaced with a 5-class linear layer.

In [29]:
model = timm.create_model(
    MODEL_NAME,
    pretrained=True,
    num_classes=NUM_CLASSES,   # replaces the 1000-class head directly
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model           : {MODEL_NAME}')
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable:,}')
print(f'Classifier head : {model.head}')

Model           : vit_base_patch16_224.augreg_in21k_ft_in1k
Total params    : 85,802,501
Trainable params: 85,802,501
Classifier head : Linear(in_features=768, out_features=5, bias=True)


## 6. Loss, Optimiser, Scheduler

Identical to the baseline — no class weights, standard AdamW + cosine schedule.

In [30]:
criterion = nn.CrossEntropyLoss()   # no class weights — matching baseline conditions
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

print('Loss      :', criterion)
print('Optimiser :', optimizer.__class__.__name__, f'lr={LR}')
print('Scheduler :', scheduler.__class__.__name__)

Loss      : CrossEntropyLoss()
Optimiser : AdamW lr=0.0003
Scheduler : CosineAnnealingLR


## 7. Training Loop

In [ ]:
use_amp = device.type == 'cuda'
scaler  = torch.amp.GradScaler('cuda', enabled=use_amp)


def run_epoch(loader, train_mode: bool):
    model.train(train_mode)
    total_loss, total = 0.0, 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, leave=False, desc='train' if train_mode else 'val'):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type='cuda', enabled=use_amp):
            logits = model(images)
            loss   = criterion(logits, labels)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item() * labels.size(0)
        total      += labels.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return total_loss / total, all_labels, all_preds


print('run_epoch defined.')

In [ ]:
best_path     = CKPT_DIR / f'{EXPERIMENT_NAME}_best.pt'
best_val_loss = float('inf')
patience_left = EARLY_STOPPING_PATIENCE

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_labels, tr_preds = run_epoch(train_loader, train_mode=True)
    vl_loss, vl_labels, vl_preds = run_epoch(val_loader,   train_mode=False)
    scheduler.step()

    tr_acc = accuracy_score(tr_labels, tr_preds)
    vl_acc = accuracy_score(vl_labels, vl_preds)

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    marker = ''
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_left = EARLY_STOPPING_PATIENCE
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_loss': vl_loss}, best_path)
        marker = '  ← best'
    else:
        patience_left -= 1

    print(f'{epoch:<8} {tr_loss:>10.4f} {tr_acc:>8.4f} {vl_loss:>10.4f} {vl_acc:>8.4f}{marker}')

    if patience_left == 0:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nBest val loss: {best_val_loss:.4f}  →  {best_path}')

## 8. Loss Curve

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

# ── Print table ────────────────────────────────────────────────
print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)
for ep, (tl, ta, vl, va) in enumerate(
    zip(history['train_loss'], history['train_acc'],
        history['val_loss'],   history['val_acc']), 1
):
    print(f'{ep:<8} {tl:>10.4f} {ta:>8.4f} {vl:>10.4f} {va:>8.4f}')
print()

# ── Plot loss curve ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs_ran, history['train_loss'], label='Train Loss', marker='o', markersize=5)
ax.plot(epochs_ran, history['val_loss'],   label='Val Loss',   marker='s', markersize=5)
ax.set_title('ViT-B/16 (IN21k) — Train vs Validation Loss', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
fig.savefig(FIGURES_DIR / f'{EXPERIMENT_NAME}_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → outputs/vit_b16/figures/{EXPERIMENT_NAME}_loss_curve.png')

## 9. Evaluation on Test Set

In [ ]:
# ── Independent of training cell — requires Sections 1–7 + checkpoint registry ─
_prev_model     = globals().get('model',     None)
_prev_criterion = globals().get('criterion', None)

model     = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()

ckpt = torch.load(CKPT_BASELINE, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'Loaded: {CKPT_BASELINE}')
print(f'epoch={ckpt["epoch"]}  val_loss={ckpt["val_loss"]:.4f}')

_, y_true, y_pred = run_epoch(test_loader, train_mode=False)

print('\nClassification Report (Test Set):')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=3))
print(f'Quadratic Weighted Kappa: {cohen_kappa_score(y_true, y_pred, weights="quadratic"):.4f}')

if _prev_model is not None:
    model     = _prev_model
    criterion = _prev_criterion


In [ ]:
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
fig.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ViT-B/16 (Test Set)', fontweight='bold')

thresh = cm.max() / 2
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f'{cm[i, j]:,}', ha='center', va='center', fontsize=9,
                color='white' if cm[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(FIGURES_DIR / f'{EXPERIMENT_NAME}_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → outputs/vit_b16/figures/{EXPERIMENT_NAME}_confusion_matrix.png')

In [ ]:
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_norm, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, format='%.2f')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ViT-B/16 (Test Set, normalised)', fontweight='bold')

thresh = 0.5
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        ax.text(j, i, f'{cm_norm[i, j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if cm_norm[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(FIGURES_DIR / f'{EXPERIMENT_NAME}_confusion_matrix_normalised.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → outputs/vit_b16/figures/{EXPERIMENT_NAME}_confusion_matrix_normalised.png')

## 10. Results Summary

In [ ]:
qwk    = cohen_kappa_score(y_true, y_pred, weights='quadratic')
acc    = accuracy_score(y_true, y_pred)
f1_per = f1_score(y_true, y_pred, average=None, labels=list(range(5)))

print('=' * 52)
print('  ViT-B/16 (ImageNet-21k) — TEST SET RESULTS')
print('=' * 52)
print(f'  Test QWK      : {qwk:.4f}')
print(f'  Test Accuracy : {acc:.4f}')
print()
print('  Per-class F1:')
for name, f1 in zip(CLASS_NAMES, f1_per):
    print(f'    {name:<18}: {f1:.4f}')
print('=' * 52)

## 11. Experiment 1 — Class Imbalance Handling

**Hypothesis:** The 36.4× class imbalance (Grade 0 = 73.5% vs Grade 4 = 2.0%) biases the baseline
towards majority classes and suppresses minority-class recall.
Combining three complementary techniques should improve QWK — especially on under-represented grades —
without changing the architecture or hyper-parameters.

**Techniques (all combined):**
| Technique | Where it acts | What it does |
|---|---|---|
| `WeightedRandomSampler` | Data pipeline | Over-samples minority grades so each epoch sees near-balanced class frequencies |
| Class-weight alpha (`α`) | Loss (per-class) | Scales the CE loss for each class by its inverse frequency |
| `FocalLoss` (γ=2) | Loss (per-sample) | Down-weights easy/well-classified examples; focuses training on hard minority samples |

> **Note:** Cells 1–10 (baseline) must be run first — this section reuses `run_epoch`, `train_ds`,
> `val_loader`, `qwk`, `acc`, and `f1_per` from those cells.

In [24]:
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight

In [25]:
class FocalLoss(nn.Module):
    """
    Focal Loss with per-class alpha weighting.
      alpha : Tensor[NUM_CLASSES] — inverse-frequency class weights (on device).
      gamma : focusing parameter; (1-pt)^gamma down-weights easy examples.
    """
    def __init__(self, alpha: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.register_buffer('alpha', alpha)  # follows .to(device); included in state_dict
        self.gamma = gamma

    def forward(self, logits: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        # Per-sample CE with class-frequency weighting (reduction='none' is required)
        ce = F.cross_entropy(logits, target, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)                       # probability assigned to the correct class
        return ((1.0 - pt) ** self.gamma * ce).mean()

print('FocalLoss defined.')

FocalLoss defined.


In [ ]:
# ── 1. Inverse-frequency class weights ───────────────────────────────────────
train_labels = train_df['level'].values

class_weights_np = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_labels,
)
class_weights_t = torch.tensor(class_weights_np, dtype=torch.float32).to(device)

print('Class weights (inverse-freq, balanced):')
for name, w in zip(CLASS_NAMES, class_weights_np):
    print(f'  {name:<20}: {w:.4f}')

# ── 2. Per-sample weights for WeightedRandomSampler ──────────────────────────
sample_weights_t = torch.as_tensor(class_weights_np[train_labels], dtype=torch.double)

exp1_sampler = WeightedRandomSampler(
    weights=sample_weights_t,
    num_samples=len(sample_weights_t),
    replacement=True,
)

# ── 3. New train loader (sampler replaces shuffle); val/test loaders unchanged ─
exp1_train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    sampler=exp1_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)

print(f'\nExp1 train loader : {len(exp1_train_loader)} batches/epoch')
print(f'Val loader (reuse): {len(val_loader)} batches/epoch')
print(f'Test loader (reuse): {len(test_loader)} batches/epoch')

In [27]:
set_seed(SEED)

# ── Fresh model (independent weights — fair comparison with baseline) ─────────
exp1_model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES)
exp1_model = exp1_model.to(device)

# ── FocalLoss: alpha = inverse-freq class weights, gamma = 2 ─────────────────
exp1_criterion = FocalLoss(alpha=class_weights_t, gamma=2.0)

# ── Same hyper-parameters as baseline ────────────────────────────────────────
exp1_optimizer = torch.optim.AdamW(exp1_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
exp1_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(exp1_optimizer, T_max=EPOCHS)
exp1_scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)

# ── Output directories ────────────────────────────────────────────────────────
EXP1_OUT_DIR  = ROOT / 'outputs' / 'vit_b16_exp1'
EXP1_CKPT_DIR = EXP1_OUT_DIR / 'checkpoints'
EXP1_FIG_DIR  = EXP1_OUT_DIR / 'figures'
for d in (EXP1_CKPT_DIR, EXP1_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

EXP1_BEST_PATH = EXP1_CKPT_DIR / 'vit_b16_exp1_best.pt'

print(f'Model     : {MODEL_NAME}')
print(f'Criterion : FocalLoss(gamma=2.0, alpha=class_weights)')
print(f'Optimiser : AdamW  lr={LR}  wd={WEIGHT_DECAY}')
print(f'Checkpts  : {EXP1_CKPT_DIR}')
print(f'Figures   : {EXP1_FIG_DIR}')

Model     : vit_base_patch16_224.augreg_in21k_ft_in1k
Criterion : FocalLoss(gamma=2.0, alpha=class_weights)
Optimiser : AdamW  lr=0.0003  wd=0.01
Checkpts  : /scratch/AML_G14_CW/outputs/vit_b16_exp1/checkpoints
Figures   : /scratch/AML_G14_CW/outputs/vit_b16_exp1/figures


In [ ]:
# ── Stash baseline bindings (safe if baseline hasn't been run) ────────────────
_baseline_model     = globals().get('model',     None)
_baseline_criterion = globals().get('criterion', None)
_baseline_optimizer = globals().get('optimizer', None)
_baseline_scaler    = globals().get('scaler',    None)

model     = exp1_model
criterion = exp1_criterion
optimizer = exp1_optimizer
scaler    = exp1_scaler

# ── Training loop (identical structure to baseline) ───────────────────────────
exp1_best_val_loss = float('inf')
exp1_patience_left = EARLY_STOPPING_PATIENCE
exp1_history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_labels, tr_preds = run_epoch(exp1_train_loader, train_mode=True)
    vl_loss, vl_labels, vl_preds = run_epoch(val_loader,        train_mode=False)
    exp1_scheduler.step()

    tr_acc = accuracy_score(tr_labels, tr_preds)
    vl_acc = accuracy_score(vl_labels, vl_preds)

    exp1_history['train_loss'].append(tr_loss)
    exp1_history['val_loss'].append(vl_loss)
    exp1_history['train_acc'].append(tr_acc)
    exp1_history['val_acc'].append(vl_acc)

    marker = ''
    if vl_loss < exp1_best_val_loss:
        exp1_best_val_loss = vl_loss
        exp1_patience_left = EARLY_STOPPING_PATIENCE
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_loss': vl_loss}, EXP1_BEST_PATH)
        marker = '  ← best'
    else:
        exp1_patience_left -= 1

    print(f'{epoch:<8} {tr_loss:>10.4f} {tr_acc:>8.4f} {vl_loss:>10.4f} {vl_acc:>8.4f}{marker}')

    if exp1_patience_left == 0:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nBest val loss: {exp1_best_val_loss:.4f}  →  {EXP1_BEST_PATH}')

# ── Restore baseline bindings ─────────────────────────────────────────────────
if _baseline_model is not None:
    model     = _baseline_model
    criterion = _baseline_criterion
    optimizer = _baseline_optimizer
    scaler    = _baseline_scaler

In [ ]:
epochs_ran_exp1 = range(1, len(exp1_history['train_loss']) + 1)

# ── Print table ───────────────────────────────────────────────────────────────
print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)
for ep, (tl, ta, vl, va) in enumerate(
    zip(exp1_history['train_loss'], exp1_history['train_acc'],
        exp1_history['val_loss'],   exp1_history['val_acc']), 1
):
    print(f'{ep:<8} {tl:>10.4f} {ta:>8.4f} {vl:>10.4f} {va:>8.4f}')
print()

# ── Plot loss curve ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs_ran_exp1, exp1_history['train_loss'], label='Train Loss', marker='o', markersize=5)
ax.plot(epochs_ran_exp1, exp1_history['val_loss'],   label='Val Loss',   marker='s', markersize=5)
ax.set_title('ViT-B/16 Exp1 — Train vs Validation Loss\n(FocalLoss γ=2 + WeightedRandomSampler)', fontweight='bold')
ax.set_xlabel('Epoch')
ax.set_ylabel('Focal Loss')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend()
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
fig.savefig(EXP1_FIG_DIR / 'vit_b16_exp1_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → outputs/vit_b16_exp1/figures/vit_b16_exp1_loss_curve.png')

In [ ]:
# ── Independent of training cell — requires Sections 1–7 + checkpoint registry ─
_prev_model     = globals().get('model',     None)
_prev_criterion = globals().get('criterion', None)

model     = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()

ckpt = torch.load(CKPT_EXP1, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'Exp1: loaded {CKPT_EXP1.name}  (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f})')

_, exp1_y_true, exp1_y_pred = run_epoch(test_loader, train_mode=False)

print('\nExp1 Classification Report (Test Set):')
print(classification_report(exp1_y_true, exp1_y_pred, target_names=CLASS_NAMES, digits=3))

if _prev_model is not None:
    model     = _prev_model
    criterion = _prev_criterion


In [ ]:
cm_exp1 = confusion_matrix(exp1_y_true, exp1_y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_exp1, interpolation='nearest', cmap='Oranges')
fig.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ViT-B/16 Exp1 (Test Set)\n(FocalLoss γ=2 + WeightedRandomSampler)', fontweight='bold')

thresh = cm_exp1.max() / 2
for i in range(cm_exp1.shape[0]):
    for j in range(cm_exp1.shape[1]):
        ax.text(j, i, f'{cm_exp1[i, j]:,}', ha='center', va='center', fontsize=9,
                color='white' if cm_exp1[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(EXP1_FIG_DIR / 'vit_b16_exp1_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → outputs/vit_b16_exp1/figures/vit_b16_exp1_confusion_matrix.png')

In [ ]:
cm_exp1_norm = cm_exp1.astype(float) / cm_exp1.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_exp1_norm, interpolation='nearest', cmap='Oranges', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, format='%.2f')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ViT-B/16 Exp1 (Test Set, normalised)\n(FocalLoss γ=2 + WeightedRandomSampler)', fontweight='bold')

thresh = 0.5
for i in range(cm_exp1_norm.shape[0]):
    for j in range(cm_exp1_norm.shape[1]):
        ax.text(j, i, f'{cm_exp1_norm[i, j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if cm_exp1_norm[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(EXP1_FIG_DIR / 'vit_b16_exp1_confusion_matrix_normalised.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → outputs/vit_b16_exp1/figures/vit_b16_exp1_confusion_matrix_normalised.png')

In [ ]:
# ── Exp1 summary metrics ───────────────────────────────────────────────────────
exp1_qwk    = cohen_kappa_score(exp1_y_true, exp1_y_pred, weights='quadratic')
exp1_acc    = accuracy_score(exp1_y_true, exp1_y_pred)
exp1_f1_per = f1_score(exp1_y_true, exp1_y_pred, average=None, labels=list(range(5)))

# ── Side-by-side comparison (baseline variables: qwk, acc, f1_per) ─────────────
print('=' * 62)
print(f'  {"Metric":<25} {"Baseline":>10}  {"Exp1 (WRS+Focal)":>16}')
print('=' * 62)
qwk_arrow = '▲' if exp1_qwk > qwk else '▼'
print(f'  {"Test QWK":<25} {qwk:>10.4f}  {exp1_qwk:>16.4f}  {qwk_arrow}')
print(f'  {"Test Accuracy":<25} {acc:>10.4f}  {exp1_acc:>16.4f}')
print()
print(f'  Per-class F1:')
for name, f1_b, f1_e in zip(CLASS_NAMES, f1_per, exp1_f1_per):
    delta = f1_e - f1_b
    print(f'    {name:<21}: {f1_b:.4f}  →  {f1_e:.4f}  ({delta:+.4f})')
print('=' * 62)
print(f'  QWK delta: {exp1_qwk - qwk:+.4f}')
print('=' * 62)

## 12. Experiment 2 — Enhanced Augmentation

**Hypothesis:** The baseline training transform (resize → crop → HFlip) omits several augmentations
that improve generalisation on retinal images. Adding RandomVerticalFlip, RandomRotation, ColorJitter,
and RandomAffine should reduce overfitting and improve QWK — especially for minority classes with few
training examples.

**Additions over Exp1:**

| Augmentation | Rationale |
|---|---|
| `RandomVerticalFlip(p=0.5)` | Fundus images have no canonical up/down orientation |
| `RandomRotation(15°)` | Rotation-invariance for vessel patterns |
| `ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02)` | Illumination robustness |
| `RandomAffine(translate=0.05, scale=0.95–1.05)` | Minor camera-position variance |

> **Carried forward from Exp1:** WeightedRandomSampler (`exp1_sampler`), FocalLoss (γ=2), `class_weights_t`.

In [ ]:
exp2_train_transform = transforms.Compose([
    transforms.Resize(int(IMAGE_SIZE * 1.14)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomApply([transforms.RandomRotation(degrees=15)], p=0.5),
    transforms.RandomApply([
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02)
    ], p=0.3),
    transforms.RandomApply([
        transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05))
    ], p=0.3),
    transforms.ToTensor(),
    transforms.Normalize(VIT_MEAN, VIT_STD),
])

exp2_train_ds = DRDataset(train_df, TRAIN_DIR, transform=exp2_train_transform)

# Reuse exp1_sampler (WeightedRandomSampler) — same class-balance strategy
exp2_train_loader = DataLoader(
    exp2_train_ds,
    batch_size=BATCH_SIZE,
    sampler=exp1_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)

print(f'Exp2 train loader: {len(exp2_train_loader)} batches/epoch  (augmented transform)')
print('Added: RandomVerticalFlip, RandomRotation(15°), ColorJitter, RandomAffine')

In [ ]:
set_seed(SEED)
exp2_model     = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(device)
exp2_criterion = FocalLoss(alpha=class_weights_t, gamma=2.0)
exp2_optimizer = torch.optim.AdamW(exp2_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
exp2_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(exp2_optimizer, T_max=EPOCHS)
exp2_scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)

EXP2_OUT_DIR  = ROOT / 'outputs' / 'vit_b16_exp2'
EXP2_CKPT_DIR = EXP2_OUT_DIR / 'checkpoints'
EXP2_FIG_DIR  = EXP2_OUT_DIR / 'figures'
for d in (EXP2_CKPT_DIR, EXP2_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

EXP2_BEST_PATH = EXP2_CKPT_DIR / 'vit_b16_exp2_best.pt'

print(f'Exp2 model ready.')
print(f'Checkpoints → {EXP2_CKPT_DIR}')
print(f'Figures     → {EXP2_FIG_DIR}')

In [ ]:
_e2_baseline_model     = globals().get('model',     None)
_e2_baseline_criterion = globals().get('criterion', None)
_e2_baseline_optimizer = globals().get('optimizer', None)
_e2_baseline_scaler    = globals().get('scaler',    None)

model, criterion, optimizer, scaler = exp2_model, exp2_criterion, exp2_optimizer, exp2_scaler

exp2_history       = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
exp2_best_val_loss = float('inf')
exp2_patience_left = EARLY_STOPPING_PATIENCE

print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_labels, tr_preds = run_epoch(exp2_train_loader, train_mode=True)
    vl_loss, vl_labels, vl_preds = run_epoch(val_loader,        train_mode=False)
    exp2_scheduler.step()

    tr_acc = accuracy_score(tr_labels, tr_preds)
    vl_acc = accuracy_score(vl_labels, vl_preds)

    exp2_history['train_loss'].append(tr_loss)
    exp2_history['val_loss'].append(vl_loss)
    exp2_history['train_acc'].append(tr_acc)
    exp2_history['val_acc'].append(vl_acc)

    marker = ''
    if vl_loss < exp2_best_val_loss:
        exp2_best_val_loss = vl_loss
        exp2_patience_left = EARLY_STOPPING_PATIENCE
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_loss': vl_loss}, EXP2_BEST_PATH)
        marker = '  ← best'
    else:
        exp2_patience_left -= 1

    print(f'{epoch:<8} {tr_loss:>10.4f} {tr_acc:>8.4f} {vl_loss:>10.4f} {vl_acc:>8.4f}{marker}')

    if exp2_patience_left == 0:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nBest val loss: {exp2_best_val_loss:.4f}  →  {EXP2_BEST_PATH}')

if _e2_baseline_model is not None:
    model, criterion, optimizer, scaler = (
        _e2_baseline_model, _e2_baseline_criterion,
        _e2_baseline_optimizer, _e2_baseline_scaler
    )

In [ ]:
epochs_ran_exp2 = range(1, len(exp2_history['train_loss']) + 1)

print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)
for ep, (tl, ta, vl, va) in enumerate(
    zip(exp2_history['train_loss'], exp2_history['train_acc'],
        exp2_history['val_loss'],   exp2_history['val_acc']), 1
):
    print(f'{ep:<8} {tl:>10.4f} {ta:>8.4f} {vl:>10.4f} {va:>8.4f}')
print()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs_ran_exp2, exp2_history['train_loss'], label='Train Loss', marker='o', markersize=5)
ax.plot(epochs_ran_exp2, exp2_history['val_loss'],   label='Val Loss',   marker='s', markersize=5)
ax.set_title('ViT-B/16 Exp2 — Train vs Validation Loss\n(FocalLoss γ=2 + WRS + Enhanced Augmentation)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Focal Loss')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend(); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(EXP2_FIG_DIR / 'vit_b16_exp2_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/vit_b16_exp2/figures/vit_b16_exp2_loss_curve.png')

In [ ]:
# ── Independent of training cell — requires Sections 1–7 + checkpoint registry ─
_prev_model     = globals().get('model',     None)
_prev_criterion = globals().get('criterion', None)

model     = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()

ckpt = torch.load(CKPT_EXP2, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'Exp2: loaded {CKPT_EXP2.name}  (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f})')

_, exp2_y_true, exp2_y_pred = run_epoch(test_loader, train_mode=False)

print('\nExp2 Classification Report (Test Set):')
print(classification_report(exp2_y_true, exp2_y_pred, target_names=CLASS_NAMES, digits=3))

if _prev_model is not None:
    model     = _prev_model
    criterion = _prev_criterion


In [ ]:
cm_exp2 = confusion_matrix(exp2_y_true, exp2_y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_exp2, interpolation='nearest', cmap='Greens')
fig.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ViT-B/16 Exp2 (Test Set)\n(FocalLoss γ=2 + WRS + Enhanced Augmentation)', fontweight='bold')

thresh = cm_exp2.max() / 2
for i in range(cm_exp2.shape[0]):
    for j in range(cm_exp2.shape[1]):
        ax.text(j, i, f'{cm_exp2[i, j]:,}', ha='center', va='center', fontsize=9,
                color='white' if cm_exp2[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(EXP2_FIG_DIR / 'vit_b16_exp2_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/vit_b16_exp2/figures/vit_b16_exp2_confusion_matrix.png')

In [ ]:
cm_exp2_norm = cm_exp2.astype(float) / cm_exp2.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_exp2_norm, interpolation='nearest', cmap='Greens', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, format='%.2f')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ViT-B/16 Exp2 (Test Set, normalised)\n(FocalLoss γ=2 + WRS + Enhanced Augmentation)', fontweight='bold')

thresh = 0.5
for i in range(cm_exp2_norm.shape[0]):
    for j in range(cm_exp2_norm.shape[1]):
        ax.text(j, i, f'{cm_exp2_norm[i, j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if cm_exp2_norm[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(EXP2_FIG_DIR / 'vit_b16_exp2_confusion_matrix_normalised.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/vit_b16_exp2/figures/vit_b16_exp2_confusion_matrix_normalised.png')

In [ ]:
exp2_qwk    = cohen_kappa_score(exp2_y_true, exp2_y_pred, weights='quadratic')
exp2_acc    = accuracy_score(exp2_y_true, exp2_y_pred)
exp2_f1_per = f1_score(exp2_y_true, exp2_y_pred, average=None, labels=list(range(5)))

print('=' * 72)
print(f'  {"Metric":<22} {"Baseline":>10}  {"Exp1 (WRS+FL)":>13}  {"Exp2 (+Aug)":>11}')
print('=' * 72)
for label, b, e1, e2 in [
    ('Test QWK',      qwk, exp1_qwk, exp2_qwk),
    ('Test Accuracy', acc, exp1_acc, exp2_acc),
]:
    arrow = '▲' if e2 > e1 else '▼'
    print(f'  {label:<22} {b:>10.4f}  {e1:>13.4f}  {e2:>11.4f}  {arrow}')
print()
print(f'  Per-class F1:')
for name, f1_b, f1_e1, f1_e2 in zip(CLASS_NAMES, f1_per, exp1_f1_per, exp2_f1_per):
    delta = f1_e2 - f1_b
    print(f'    {name:<20}: {f1_b:.4f}  {f1_e1:.4f}  {f1_e2:.4f}  ({delta:+.4f})')
print('=' * 72)
print(f'  QWK delta vs Baseline:  Exp1={exp1_qwk-qwk:+.4f}  Exp2={exp2_qwk-qwk:+.4f}')
print('=' * 72)

## 13. Experiment 3 — Two-Phase Training (Freeze → Unfreeze)

**Hypothesis:** Training all layers from epoch 1 risks distorting pre-trained ViT features before the
5-class head has learnt a reasonable mapping. Freezing the backbone first (Phase 1) lets the head
stabilise; then unfreezing everything (Phase 2) at a lower LR with gradient clipping achieves
better fine-tuning without catastrophic forgetting.

**Training protocol:**

| Phase | Backbone | Epochs | LR | Grad Clip |
|---|---|---|---|---|
| Phase 1 | Frozen (head only) | 5 | 3e-4 | — |
| Phase 2 | Unfrozen (all layers) | 5 | 3e-5 | 1.0 |

> **Cumulative:** Builds on Exp2 augmentations + Exp1 FocalLoss + WeightedRandomSampler.  
> Reuses `exp2_train_loader` and `class_weights_t` from prior cells.

In [ ]:
PHASE1_EPOCHS = 5
PHASE2_EPOCHS = 5
PHASE1_LR     = 3e-4
PHASE2_LR     = 3e-5
GRAD_CLIP     = 1.0

set_seed(SEED)
exp3_model     = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(device)
exp3_criterion = FocalLoss(alpha=class_weights_t, gamma=2.0)
exp3_scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)

EXP3_OUT_DIR  = ROOT / 'outputs' / 'vit_b16_exp3'
EXP3_CKPT_DIR = EXP3_OUT_DIR / 'checkpoints'
EXP3_FIG_DIR  = EXP3_OUT_DIR / 'figures'
for d in (EXP3_CKPT_DIR, EXP3_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

EXP3_PHASE1_PATH = EXP3_CKPT_DIR / 'vit_b16_exp3_phase1_best.pt'
EXP3_BEST_PATH   = EXP3_CKPT_DIR / 'vit_b16_exp3_best.pt'


def run_epoch_clipped(loader, train_mode: bool, grad_clip: float = 0.0):
    """run_epoch variant that supports gradient clipping (needed for Phase 2 fine-tuning)."""
    model.train(train_mode)
    total_loss, total = 0.0, 0
    all_preds, all_labels = [], []

    for images, labels in tqdm(loader, leave=False, desc='train' if train_mode else 'eval'):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.autocast(device_type='cuda', enabled=use_amp):
            logits = model(images)
            loss   = criterion(logits, labels)

        if train_mode:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            if grad_clip > 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            scaler.step(optimizer)
            scaler.update()

        total_loss += loss.item() * labels.size(0)
        total      += labels.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    return total_loss / total, all_labels, all_preds


print(f'Exp3 ready. Phase1 LR={PHASE1_LR}, Phase2 LR={PHASE2_LR}, GRAD_CLIP={GRAD_CLIP}')
print(f'Checkpoints → {EXP3_CKPT_DIR}')

In [ ]:
# ── Freeze backbone — only head.weight / head.bias remain trainable ───────────
for name, param in exp3_model.named_parameters():
    if 'head' not in name:
        param.requires_grad = False

trainable_p1 = sum(p.numel() for p in exp3_model.parameters() if p.requires_grad)
print(f'Phase 1: {trainable_p1:,} trainable parameters (head only)\n')

_e3_baseline_model     = globals().get('model',     None)
_e3_baseline_criterion = globals().get('criterion', None)
_e3_baseline_optimizer = globals().get('optimizer', None)
_e3_baseline_scaler    = globals().get('scaler',    None)

model     = exp3_model
criterion = exp3_criterion
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=PHASE1_LR, weight_decay=WEIGHT_DECAY,
)
scaler = exp3_scaler

scheduler_p1 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE1_EPOCHS)

exp3_history = {
    'phase1_train': [], 'phase1_val': [], 'phase1_train_acc': [], 'phase1_val_acc': [],
    'phase2_train': [], 'phase2_val': [], 'phase2_train_acc': [], 'phase2_val_acc': [],
}
p1_best_val  = float('inf')
p1_patience  = EARLY_STOPPING_PATIENCE

print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)

for epoch in range(1, PHASE1_EPOCHS + 1):
    tr_loss, tr_labels, tr_preds = run_epoch_clipped(exp2_train_loader, train_mode=True,  grad_clip=0.0)
    vl_loss, vl_labels, vl_preds = run_epoch_clipped(val_loader,        train_mode=False)
    scheduler_p1.step()

    tr_acc = accuracy_score(tr_labels, tr_preds)
    vl_acc = accuracy_score(vl_labels, vl_preds)

    exp3_history['phase1_train'].append(tr_loss)
    exp3_history['phase1_val'].append(vl_loss)
    exp3_history['phase1_train_acc'].append(tr_acc)
    exp3_history['phase1_val_acc'].append(vl_acc)

    marker = ''
    if vl_loss < p1_best_val:
        p1_best_val = vl_loss
        p1_patience = EARLY_STOPPING_PATIENCE
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_loss': vl_loss}, EXP3_PHASE1_PATH)
        marker = '  ← best'
    else:
        p1_patience -= 1

    print(f'{epoch:<8} {tr_loss:>10.4f} {tr_acc:>8.4f} {vl_loss:>10.4f} {vl_acc:>8.4f}{marker}')

    if p1_patience == 0:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nPhase 1 complete. Best val loss: {p1_best_val:.4f}')

In [ ]:
# ── Unfreeze all parameters for full fine-tuning ──────────────────────────────
for param in exp3_model.parameters():
    param.requires_grad = True

trainable_p2 = sum(p.numel() for p in exp3_model.parameters() if p.requires_grad)
print(f'Phase 2: {trainable_p2:,} trainable parameters (all layers)\n')

# Start from Phase 1 best weights
ckpt_p1 = torch.load(EXP3_PHASE1_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt_p1['model_state'])
print(f'Loaded Phase 1 best (epoch {ckpt_p1["epoch"]}, val_loss={ckpt_p1["val_loss"]:.4f})')

optimizer    = torch.optim.AdamW(model.parameters(), lr=PHASE2_LR, weight_decay=WEIGHT_DECAY)
scheduler_p2 = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE2_EPOCHS)

p2_best_val = float('inf')
p2_patience = EARLY_STOPPING_PATIENCE

print(f"\n{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)

for epoch in range(1, PHASE2_EPOCHS + 1):
    tr_loss, tr_labels, tr_preds = run_epoch_clipped(exp2_train_loader, train_mode=True, grad_clip=GRAD_CLIP)
    vl_loss, vl_labels, vl_preds = run_epoch_clipped(val_loader,        train_mode=False)
    scheduler_p2.step()

    tr_acc = accuracy_score(tr_labels, tr_preds)
    vl_acc = accuracy_score(vl_labels, vl_preds)

    exp3_history['phase2_train'].append(tr_loss)
    exp3_history['phase2_val'].append(vl_loss)
    exp3_history['phase2_train_acc'].append(tr_acc)
    exp3_history['phase2_val_acc'].append(vl_acc)

    marker = ''
    if vl_loss < p2_best_val:
        p2_best_val = vl_loss
        p2_patience = EARLY_STOPPING_PATIENCE
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_loss': vl_loss}, EXP3_BEST_PATH)
        marker = '  ← best'
    else:
        p2_patience -= 1

    print(f'{epoch:<8} {tr_loss:>10.4f} {tr_acc:>8.4f} {vl_loss:>10.4f} {vl_acc:>8.4f}{marker}')

    if p2_patience == 0:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nPhase 2 complete. Best val loss: {p2_best_val:.4f}  →  {EXP3_BEST_PATH}')

if _e3_baseline_model is not None:
    model, criterion, optimizer, scaler = (
        _e3_baseline_model, _e3_baseline_criterion,
        _e3_baseline_optimizer, _e3_baseline_scaler
    )

In [ ]:
p1_epochs = range(1, len(exp3_history['phase1_train']) + 1)
p2_start  = len(exp3_history['phase1_train'])
p2_epochs = range(p2_start + 1, p2_start + len(exp3_history['phase2_train']) + 1)

print('Phase 1:')
print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)
for ep, (tl, ta, vl, va) in enumerate(
    zip(exp3_history['phase1_train'], exp3_history['phase1_train_acc'],
        exp3_history['phase1_val'],   exp3_history['phase1_val_acc']), 1
):
    print(f'{ep:<8} {tl:>10.4f} {ta:>8.4f} {vl:>10.4f} {va:>8.4f}')

print('\nPhase 2:')
print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)
for ep, (tl, ta, vl, va) in enumerate(
    zip(exp3_history['phase2_train'], exp3_history['phase2_train_acc'],
        exp3_history['phase2_val'],   exp3_history['phase2_val_acc']), p2_start + 1
):
    print(f'{ep:<8} {tl:>10.4f} {ta:>8.4f} {vl:>10.4f} {va:>8.4f}')
print()

all_train  = exp3_history['phase1_train'] + exp3_history['phase2_train']
all_val    = exp3_history['phase1_val']   + exp3_history['phase2_val']
all_epochs = range(1, len(all_train) + 1)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(all_epochs, all_train, label='Train Loss', marker='o', markersize=5)
ax.plot(all_epochs, all_val,   label='Val Loss',   marker='s', markersize=5)
if p2_start > 0:
    ax.axvline(x=p2_start + 0.5, color='gray', linestyle='--', linewidth=1, label='Phase 2 start (unfreeze)')
ax.set_title('ViT-B/16 Exp3 — Two-Phase Training Loss\n(Phase 1: head only → Phase 2: full fine-tune + grad clip)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Focal Loss')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend(); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(EXP3_FIG_DIR / 'vit_b16_exp3_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/vit_b16_exp3/figures/vit_b16_exp3_loss_curve.png')

In [ ]:
# ── Independent of training cell — requires Sections 1–7 + checkpoint registry ─
# Uses run_epoch (not run_epoch_clipped) — grad clipping is irrelevant for eval.
_prev_model     = globals().get('model',     None)
_prev_criterion = globals().get('criterion', None)

model     = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()

ckpt = torch.load(CKPT_EXP3, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'Exp3: loaded {CKPT_EXP3.name}  (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f})')

_, exp3_y_true, exp3_y_pred = run_epoch(test_loader, train_mode=False)

print('\nExp3 Classification Report (Test Set):')
print(classification_report(exp3_y_true, exp3_y_pred, target_names=CLASS_NAMES, digits=3))

if _prev_model is not None:
    model     = _prev_model
    criterion = _prev_criterion


In [ ]:
cm_exp3 = confusion_matrix(exp3_y_true, exp3_y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_exp3, interpolation='nearest', cmap='Purples')
fig.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ViT-B/16 Exp3 (Test Set)\n(Two-Phase: Frozen → Unfreeze + Grad Clip)', fontweight='bold')

thresh = cm_exp3.max() / 2
for i in range(cm_exp3.shape[0]):
    for j in range(cm_exp3.shape[1]):
        ax.text(j, i, f'{cm_exp3[i, j]:,}', ha='center', va='center', fontsize=9,
                color='white' if cm_exp3[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(EXP3_FIG_DIR / 'vit_b16_exp3_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/vit_b16_exp3/figures/vit_b16_exp3_confusion_matrix.png')

In [ ]:
cm_exp3_norm = cm_exp3.astype(float) / cm_exp3.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_exp3_norm, interpolation='nearest', cmap='Purples', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, format='%.2f')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ViT-B/16 Exp3 (Test Set, normalised)\n(Two-Phase: Frozen → Unfreeze + Grad Clip)', fontweight='bold')

thresh = 0.5
for i in range(cm_exp3_norm.shape[0]):
    for j in range(cm_exp3_norm.shape[1]):
        ax.text(j, i, f'{cm_exp3_norm[i, j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if cm_exp3_norm[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(EXP3_FIG_DIR / 'vit_b16_exp3_confusion_matrix_normalised.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/vit_b16_exp3/figures/vit_b16_exp3_confusion_matrix_normalised.png')

In [ ]:
exp3_qwk    = cohen_kappa_score(exp3_y_true, exp3_y_pred, weights='quadratic')
exp3_acc    = accuracy_score(exp3_y_true, exp3_y_pred)
exp3_f1_per = f1_score(exp3_y_true, exp3_y_pred, average=None, labels=list(range(5)))

W = 82
print('=' * W)
print(f'  {"Metric":<22} {"Baseline":>10}  {"Exp1 (WRS+FL)":>13}  {"Exp2 (+Aug)":>11}  {"Exp3 (2-Phase)":>14}')
print('=' * W)
for label, b, e1, e2, e3 in [
    ('Test QWK',      qwk, exp1_qwk, exp2_qwk, exp3_qwk),
    ('Test Accuracy', acc, exp1_acc, exp2_acc, exp3_acc),
]:
    print(f'  {label:<22} {b:>10.4f}  {e1:>13.4f}  {e2:>11.4f}  {e3:>14.4f}')
print()
print(f'  Per-class F1:')
for name, f1_b, f1_e1, f1_e2, f1_e3 in zip(CLASS_NAMES, f1_per, exp1_f1_per, exp2_f1_per, exp3_f1_per):
    print(f'    {name:<20}: {f1_b:.4f}  {f1_e1:.4f}  {f1_e2:.4f}  {f1_e3:.4f}')
print('=' * W)
print(f'  QWK delta vs Baseline:  Exp1={exp1_qwk-qwk:+.4f}  Exp2={exp2_qwk-qwk:+.4f}  Exp3={exp3_qwk-qwk:+.4f}')
print('=' * W)

## 14. Experiment 4 — Hyperparameter Grid Search (LR × Batch Size × Weight Decay)

**Goal:** Identify the optimal learning rate, batch size, and weight decay before committing to full-length runs.  
Stochastic depth (`drop_path_rate`) is fixed at 0.1 — a safe default from the ViT paper that adds light regularisation to transformer blocks.

**Search space:**

| Hyperparameter | Values | Notes |
|---|---|---|
| Learning rate | 1e-4, 3e-4, 1e-3 | Cosine-decayed from epoch 1 |
| Batch size | 16, 32, 64 | Affects gradient noise and steps/epoch |
| Weight decay | 1e-3, 1e-2 | AdamW L2 penalty |
| `drop_path_rate` | **0.1 (fixed)** | Stochastic depth for transformer blocks |

**Protocol:** Each of the 18 combinations is trained for up to 5 epochs (early stopping patience=2)
using the Exp2 setup — FocalLoss (γ=2) + WeightedRandomSampler + enhanced augmentation.
Val loss selects the winner. No test-set access during search.

> **Expected runtime:** ~90–120 min.  
> **Dependencies:** Sections 11–12 must be run first (`exp2_train_ds`, `exp1_sampler`, `class_weights_t`, `FocalLoss`).

In [ ]:
TUNE_LRS          = [1e-4, 3e-4, 1e-3]
TUNE_BATCH_SIZES  = [16, 32, 64]
TUNE_WEIGHT_DECAYS = [1e-3, 1e-2]
TUNE_DROP_PATH    = 0.1   # stochastic depth — fixed across the search
TUNE_EPOCHS       = 5
TUNE_PATIENCE     = 2

n_combos = len(TUNE_LRS) * len(TUNE_BATCH_SIZES) * len(TUNE_WEIGHT_DECAYS)

EXP4_OUT_DIR  = ROOT / 'outputs' / 'vit_b16_exp4'
EXP4_CKPT_DIR = EXP4_OUT_DIR / 'checkpoints'
EXP4_FIG_DIR  = EXP4_OUT_DIR / 'figures'
for d in (EXP4_CKPT_DIR, EXP4_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f'Grid: {len(TUNE_LRS)} LRs × {len(TUNE_BATCH_SIZES)} BSes × {len(TUNE_WEIGHT_DECAYS)} WDs = {n_combos} combinations')
print(f'Max epochs per run : {TUNE_EPOCHS}  (early stop patience={TUNE_PATIENCE})')
print(f'drop_path_rate     : {TUNE_DROP_PATH} (fixed)')
print(f'Checkpoints        → {EXP4_CKPT_DIR}')

In [ ]:
tune_results = []
combo_num    = 0

print(f"{'#':<4} {'LR':>8} {'BS':>4} {'WD':>8}  {'Best Ep':>7} {'Val Loss':>10} {'Val Acc':>9}")
print('─' * 58)

for bs in TUNE_BATCH_SIZES:
    tune_train_loader_bs = DataLoader(
        exp2_train_ds, batch_size=bs, sampler=exp1_sampler,
        num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
        persistent_workers=NUM_WORKERS > 0,
    )
    tune_val_loader_bs = DataLoader(
        val_ds, batch_size=bs, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=True,
        persistent_workers=NUM_WORKERS > 0,
    )

    for wd in TUNE_WEIGHT_DECAYS:
        for lr in TUNE_LRS:
            combo_num += 1
            set_seed(SEED)

            tune_model = timm.create_model(
                MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES,
                drop_path_rate=TUNE_DROP_PATH,
            ).to(device)
            tune_criterion = FocalLoss(alpha=class_weights_t, gamma=2.0)
            tune_optimizer = torch.optim.AdamW(tune_model.parameters(), lr=lr, weight_decay=wd)
            tune_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(tune_optimizer, T_max=TUNE_EPOCHS)
            tune_scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)

            # Rebind free variables used by run_epoch
            _t_model, _t_crit, _t_opt, _t_sc = model, criterion, optimizer, scaler
            model, criterion, optimizer, scaler = tune_model, tune_criterion, tune_optimizer, tune_scaler

            best_loss     = float('inf')
            best_acc      = 0.0
            best_epoch    = 0
            tune_patience = TUNE_PATIENCE

            for epoch in range(1, TUNE_EPOCHS + 1):
                run_epoch(tune_train_loader_bs, train_mode=True)
                ep_loss, ep_labels, ep_preds = run_epoch(tune_val_loader_bs, train_mode=False)
                tune_scheduler.step()

                ep_acc = accuracy_score(ep_labels, ep_preds)
                if ep_loss < best_loss:
                    best_loss, best_acc, best_epoch = ep_loss, ep_acc, epoch
                    tune_patience = TUNE_PATIENCE
                    torch.save(
                        {'epoch': epoch, 'model_state': tune_model.state_dict(), 'val_loss': ep_loss},
                        EXP4_CKPT_DIR / f'tune_bs{bs}_lr{lr:.0e}_wd{wd:.0e}_best.pt',
                    )
                else:
                    tune_patience -= 1
                if tune_patience == 0:
                    break

            print(f'{combo_num:<4} {lr:>8.0e} {bs:>4} {wd:>8.0e}   ep={best_epoch:2d}    {best_loss:>10.4f}  {best_acc:>9.4f}')
            tune_results.append({
                'lr': lr, 'batch_size': bs, 'weight_decay': wd,
                'best_val_loss': best_loss, 'best_val_acc': best_acc,
                'stopped_at': best_epoch,
            })

            # Restore and free CUDA memory before next run
            model, criterion, optimizer, scaler = _t_model, _t_crit, _t_opt, _t_sc
            del tune_model
            torch.cuda.empty_cache()

print('\nGrid search complete.')

In [ ]:
import seaborn as sns

tune_df = pd.DataFrame(tune_results).sort_values('best_val_loss').reset_index(drop=True)

# ── Ranked results table ───────────────────────────────────────────────────────
print('Grid Search Results (ranked by val loss):')
print(f"\n{'Rank':<6} {'LR':>8} {'BS':>4} {'WD':>8} {'Stop Ep':>8} {'Val Loss':>10} {'Val Acc':>9}")
print('─' * 62)
for rank, row in tune_df.iterrows():
    marker = '  ← BEST' if rank == 0 else ''
    print(f'{rank+1:<6} {row["lr"]:>8.0e} {int(row["batch_size"]):>4} '
          f'{row["weight_decay"]:>8.0e} {int(row["stopped_at"]):>8} '
          f'{row["best_val_loss"]:>10.4f} {row["best_val_acc"]:>9.4f}{marker}')

best_row = tune_df.iloc[0]
best_lr  = best_row['lr']
best_bs  = int(best_row['batch_size'])
best_wd  = best_row['weight_decay']
print(f'\n→ Best config: LR={best_lr:.0e}  Batch size={best_bs}  Weight decay={best_wd:.0e}')
print(  f'  Val loss={best_row["best_val_loss"]:.4f}  Val acc={best_row["best_val_acc"]:.4f}')

# ── Heatmaps — (rows = weight_decay, cols = val loss / val accuracy) ──────────
wd_values = sorted(tune_df['weight_decay'].unique())
fig, axes = plt.subplots(len(wd_values), 2, figsize=(14, 5 * len(wd_values)))

for row_idx, wd in enumerate(wd_values):
    subset     = tune_df[tune_df['weight_decay'] == wd]
    pivot_loss = (subset.pivot(index='batch_size', columns='lr', values='best_val_loss')
                        .sort_index(ascending=False))
    pivot_acc  = (subset.pivot(index='batch_size', columns='lr', values='best_val_acc')
                        .sort_index(ascending=False))

    for col_idx, (data, title, cmap) in enumerate([
        (pivot_loss, f'Val Loss (↓ better)  |  WD={wd:.0e}',     'RdYlGn_r'),
        (pivot_acc,  f'Val Accuracy (↑ better)  |  WD={wd:.0e}', 'RdYlGn'),
    ]):
        ax = axes[row_idx][col_idx]
        sns.heatmap(data, annot=True, fmt='.4f', cmap=cmap, ax=ax,
                    linewidths=0.5, annot_kws={'size': 11})
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Learning Rate')
        ax.set_ylabel('Batch Size')
        ax.set_xticklabels([f'{float(c):.0e}' for c in data.columns])

plt.suptitle('Hyperparameter Grid Search — LR × Batch Size × Weight Decay\n(ViT-B/16, Exp2 setup, 5-epoch probe)',
             fontweight='bold', y=1.02)
plt.tight_layout()
fig.savefig(EXP4_FIG_DIR / 'vit_b16_exp4_hparam_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/vit_b16_exp4/figures/vit_b16_exp4_hparam_heatmap.png')
print(f'\nUse LR={best_lr:.0e}, batch_size={best_bs}, weight_decay={best_wd:.0e} for subsequent full training runs.')


## 15. Experiment 5 — Majority-Class Downsampling

**Hypothesis:** Grade 0 (No DR) accounts for ~73.5% of the training set, overwhelming the model
with easy negatives. Simply removing the excess — capping Grade 0 at the size of the next-largest
class (Grade 2) — should break the dominant imbalance while preserving every rare-grade sample.

**Design:** This experiment deliberately uses standard `CrossEntropyLoss` with no sampler or
class weights, so the result isolates the effect of data-level rebalancing alone.

| Class | Original count | Exp 5 count |
|---|---|---|
| Grade 0 — No DR | ~20,580 | capped to Grade 2 count |
| Grade 1 — Mild | ~1,960 | unchanged |
| Grade 2 — Moderate | ~4,200 | unchanged |
| Grade 3 — Severe | ~700 | unchanged |
| Grade 4 — Proliferative | ~560 | unchanged |

**Comparison point vs Exp 1:** Both target the same imbalance problem; Exp 1 keeps all data but
reweights via `WeightedRandomSampler` + `FocalLoss`, while Exp 5 removes the excess at source
and lets the balanced distribution do the work.

> **Dependencies:** Sections 1–10 must be run first (`train_df`, `DRDataset`, `train_transform`, `val_loader`, `test_loader`).

In [ ]:
# ── Cap Grade 0 at Grade 2's count ───────────────────────────────────────────
grade2_count  = (train_df['level'] == 2).sum()
grade0_capped = (train_df[train_df['level'] == 0]
                 .sample(n=grade2_count, random_state=SEED))
exp5_train_df = (pd.concat([grade0_capped, train_df[train_df['level'] != 0]])
                 .sample(frac=1, random_state=SEED)
                 .reset_index(drop=True))

print(f'Original training set : {len(train_df):,} images')
print(f'Exp5 training set     : {len(exp5_train_df):,} images')
print(f'\nGrade 0 cap target    : {grade2_count:,}  (= Grade 2 count)\n')
print(f'{"Grade":<6} {"Name":<18} {"Original":>10} {"Exp 5":>8}')
print('-' * 46)
for g, name in enumerate(CLASS_NAMES):
    orig = (train_df['level'] == g).sum()
    new  = (exp5_train_df['level'] == g).sum()
    note = '  ← capped' if g == 0 else ''
    print(f'  {g}    {name:<18} {orig:>9,}  {new:>7,}{note}')

exp5_train_ds = DRDataset(exp5_train_df, TRAIN_DIR, transform=train_transform)
exp5_train_loader = DataLoader(
    exp5_train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True,
    persistent_workers=NUM_WORKERS > 0,
)
print(f'\nExp5 train loader: {len(exp5_train_loader)} batches/epoch')


In [ ]:
set_seed(SEED)
exp5_model     = timm.create_model(MODEL_NAME, pretrained=True, num_classes=NUM_CLASSES).to(device)
exp5_criterion = nn.CrossEntropyLoss()
exp5_optimizer = torch.optim.AdamW(exp5_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
exp5_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(exp5_optimizer, T_max=EPOCHS)
exp5_scaler    = torch.amp.GradScaler('cuda', enabled=use_amp)

EXP5_OUT_DIR  = ROOT / 'outputs' / 'vit_b16_exp5'
EXP5_CKPT_DIR = EXP5_OUT_DIR / 'checkpoints'
EXP5_FIG_DIR  = EXP5_OUT_DIR / 'figures'
for d in (EXP5_CKPT_DIR, EXP5_FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

EXP5_BEST_PATH = EXP5_CKPT_DIR / 'vit_b16_exp5_best.pt'

print('Exp5 model ready.')
print('Loss      : CrossEntropyLoss (no class weights — downsampling provides the balance)')
print(f'Optimiser : AdamW  lr={LR}  wd={WEIGHT_DECAY}')
print(f'Checkpts  → {EXP5_CKPT_DIR}')
print(f'Figures   → {EXP5_FIG_DIR}')


In [ ]:
_e5_baseline_model     = globals().get('model',     None)
_e5_baseline_criterion = globals().get('criterion', None)
_e5_baseline_optimizer = globals().get('optimizer', None)
_e5_baseline_scaler    = globals().get('scaler',    None)

model, criterion, optimizer, scaler = exp5_model, exp5_criterion, exp5_optimizer, exp5_scaler

exp5_history       = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
exp5_best_val_loss = float('inf')
exp5_patience_left = EARLY_STOPPING_PATIENCE

print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_labels, tr_preds = run_epoch(exp5_train_loader, train_mode=True)
    vl_loss, vl_labels, vl_preds = run_epoch(val_loader,        train_mode=False)
    exp5_scheduler.step()

    tr_acc = accuracy_score(tr_labels, tr_preds)
    vl_acc = accuracy_score(vl_labels, vl_preds)

    exp5_history['train_loss'].append(tr_loss)
    exp5_history['val_loss'].append(vl_loss)
    exp5_history['train_acc'].append(tr_acc)
    exp5_history['val_acc'].append(vl_acc)

    marker = ''
    if vl_loss < exp5_best_val_loss:
        exp5_best_val_loss = vl_loss
        exp5_patience_left = EARLY_STOPPING_PATIENCE
        torch.save({'epoch': epoch, 'model_state': model.state_dict(),
                    'val_loss': vl_loss}, EXP5_BEST_PATH)
        marker = '  ← best'
    else:
        exp5_patience_left -= 1

    print(f'{epoch:<8} {tr_loss:>10.4f} {tr_acc:>8.4f} {vl_loss:>10.4f} {vl_acc:>8.4f}{marker}')

    if exp5_patience_left == 0:
        print(f'\nEarly stopping at epoch {epoch}.')
        break

print(f'\nBest val loss: {exp5_best_val_loss:.4f}  →  {EXP5_BEST_PATH}')

if _e5_baseline_model is not None:
    model, criterion, optimizer, scaler = (
        _e5_baseline_model, _e5_baseline_criterion,
        _e5_baseline_optimizer, _e5_baseline_scaler
    )


In [ ]:
epochs_ran_exp5 = range(1, len(exp5_history['train_loss']) + 1)

print(f"{'Epoch':<8} {'Tr Loss':>10} {'Tr Acc':>8} {'Val Loss':>10} {'Val Acc':>8}")
print('─' * 50)
for ep, (tl, ta, vl, va) in enumerate(
    zip(exp5_history['train_loss'], exp5_history['train_acc'],
        exp5_history['val_loss'],   exp5_history['val_acc']), 1
):
    print(f'{ep:<8} {tl:>10.4f} {ta:>8.4f} {vl:>10.4f} {va:>8.4f}')
print()

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(epochs_ran_exp5, exp5_history['train_loss'], label='Train Loss', marker='o', markersize=5)
ax.plot(epochs_ran_exp5, exp5_history['val_loss'],   label='Val Loss',   marker='s', markersize=5)
ax.set_title('ViT-B/16 Exp5 — Train vs Validation Loss\n(Grade 0 downsampled to Grade 2 count, CE loss)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('Cross-Entropy Loss')
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax.legend(); ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
fig.savefig(EXP5_FIG_DIR / 'vit_b16_exp5_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/vit_b16_exp5/figures/vit_b16_exp5_loss_curve.png')


In [ ]:
# ── Independent of training cell — requires Sections 1–7 + checkpoint registry ─
_prev_model     = globals().get('model',     None)
_prev_criterion = globals().get('criterion', None)

model     = timm.create_model(MODEL_NAME, pretrained=False, num_classes=NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()

ckpt = torch.load(CKPT_EXP5, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state'])
print(f'Exp5: loaded {CKPT_EXP5.name}  (epoch {ckpt["epoch"]}, val_loss={ckpt["val_loss"]:.4f})')

_, exp5_y_true, exp5_y_pred = run_epoch(test_loader, train_mode=False)

print('\nExp5 Classification Report (Test Set):')
print(classification_report(exp5_y_true, exp5_y_pred, target_names=CLASS_NAMES, digits=3))

if _prev_model is not None:
    model     = _prev_model
    criterion = _prev_criterion


In [ ]:
cm_exp5 = confusion_matrix(exp5_y_true, exp5_y_pred)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_exp5, interpolation='nearest', cmap='YlOrRd')
fig.colorbar(im, ax=ax)
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ViT-B/16 Exp5 (Test Set)\n(Grade 0 downsampled to Grade 2 count)', fontweight='bold')

thresh = cm_exp5.max() / 2
for i in range(cm_exp5.shape[0]):
    for j in range(cm_exp5.shape[1]):
        ax.text(j, i, f'{cm_exp5[i, j]:,}', ha='center', va='center', fontsize=9,
                color='white' if cm_exp5[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(EXP5_FIG_DIR / 'vit_b16_exp5_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/vit_b16_exp5/figures/vit_b16_exp5_confusion_matrix.png')


In [ ]:
cm_exp5_norm = cm_exp5.astype(float) / cm_exp5.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm_exp5_norm, interpolation='nearest', cmap='YlOrRd', vmin=0, vmax=1)
fig.colorbar(im, ax=ax, format='%.2f')
ax.set_xticks(range(NUM_CLASSES)); ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_yticks(range(NUM_CLASSES)); ax.set_yticklabels(CLASS_NAMES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — ViT-B/16 Exp5 (Test Set, normalised)\n(Grade 0 downsampled to Grade 2 count)', fontweight='bold')

thresh = 0.5
for i in range(cm_exp5_norm.shape[0]):
    for j in range(cm_exp5_norm.shape[1]):
        ax.text(j, i, f'{cm_exp5_norm[i, j]:.2f}', ha='center', va='center', fontsize=9,
                color='white' if cm_exp5_norm[i, j] > thresh else 'black')

plt.tight_layout()
fig.savefig(EXP5_FIG_DIR / 'vit_b16_exp5_confusion_matrix_normalised.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → outputs/vit_b16_exp5/figures/vit_b16_exp5_confusion_matrix_normalised.png')


In [ ]:
exp5_qwk    = cohen_kappa_score(exp5_y_true, exp5_y_pred, weights='quadratic')
exp5_acc    = accuracy_score(exp5_y_true, exp5_y_pred)
exp5_f1_per = f1_score(exp5_y_true, exp5_y_pred, average=None, labels=list(range(5)))

W = 96
print('=' * W)
print(f'  {"Metric":<22} {"Baseline":>10}  {"Exp1 (WRS+FL)":>13}  {"Exp2 (+Aug)":>11}  {"Exp3 (2-Phase)":>14}  {"Exp5 (Downsample)":>17}')
print('=' * W)
for label, b, e1, e2, e3, e5 in [
    ('Test QWK',      qwk,  exp1_qwk, exp2_qwk, exp3_qwk, exp5_qwk),
    ('Test Accuracy', acc,  exp1_acc, exp2_acc, exp3_acc, exp5_acc),
]:
    print(f'  {label:<22} {b:>10.4f}  {e1:>13.4f}  {e2:>11.4f}  {e3:>14.4f}  {e5:>17.4f}')
print()
print(f'  Per-class F1:')
for name, f1_b, f1_e1, f1_e2, f1_e3, f1_e5 in zip(CLASS_NAMES, f1_per, exp1_f1_per, exp2_f1_per, exp3_f1_per, exp5_f1_per):
    print(f'    {name:<20}: {f1_b:.4f}  {f1_e1:.4f}  {f1_e2:.4f}  {f1_e3:.4f}  {f1_e5:.4f}')
print('=' * W)
print(f'  QWK delta vs Baseline:  Exp1={exp1_qwk-qwk:+.4f}  Exp2={exp2_qwk-qwk:+.4f}  Exp3={exp3_qwk-qwk:+.4f}  Exp5={exp5_qwk-qwk:+.4f}')
print('=' * W)
